### Requirements

To run `fwhm_features.ipynb`, make sure the following files are present in the same directory:

- `Feature_generator.xlsx`
- `elements.xlsx`
- CIF files (`*.cif`)

## Generate Compositional Features

In [4]:
import pandas as pd
import numpy as np
from pymatgen.core import Composition


class FormulaVectorizer:

    def __init__(self, element_file="elements.xlsx"):
        self.df = pd.read_excel(element_file, index_col="Symbol")
        self.df.columns = self.df.columns.str.strip()

        self.feature_names = [
            f"{s}_{p}"
            for s in ["avg", "diff", "max", "min", "std"]
            for p in self.df.columns
        ]

    def featurize(self, formula):
        try:
            comp = Composition(str(formula)).fractional_composition.as_dict()
            elems = [e for e in comp if e in self.df.index]

            avg = sum(self.df.loc[e] * comp[e] for e in elems)
            props = self.df.loc[elems]

            return np.concatenate([
                avg.values,
                (props.max() - props.min()).values,
                props.max().values,
                props.min().values,
                props.std(ddof=0).values,
            ])

        except:
            return [np.nan] * len(self.feature_names)


df = pd.read_excel("Feature_generator.xlsx")
vf = FormulaVectorizer("elements.xlsx")

feature_df = pd.DataFrame(
    [vf.featurize(f) for f in df["Formula"]],
    columns=vf.feature_names
)

columns = [
    "Formula",
    "avg_Gordy EN",
    "polyhedron volume",
    "avg_Mulliken EN",
    "max_metal_ligand_bond_length",
    "std_Heat of fusion (kJ/mol)",
    "mean_metal_ligand_bond_length",
    "avg_First ionization energy (kJ/mol)",
    "R5",
    "diff_Thermal conductivity (W/m•K)",
    "Vol/fu",
    "x",
    "avg_Heat of vaporization (kJ/mol)",
    "avg_Martynov-Batsanov EN",
    "avg_Pauling EN",
    "std_Mulliken EN",
    "std_Heat atomization (kJ/mol)",
    "avg_Number of p electrons",
    "std_Thermal conductivity (W/m•K)"
]

out = pd.DataFrame({"Formula": df["Formula"]})

for c in columns[1:]:
    out[c] = feature_df[c] if c in feature_df else np.nan

out = out[columns]
out.to_excel("To_predict_fwhm.xlsx", index=False)

print("Saved: To_predict_fwhm.xlsx")

Saved: To_predict_fwhm.xlsx


## ### Next Step

Use the generated `To_predict_fwhm.xlsx` file for final prediction after adding the remaining structural features

## Extract Structural Features from CIF Files

In [6]:
import os
import pandas as pd
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

rows = []

for cif in [f for f in os.listdir() if f.endswith(".cif")]:

    try:
        s = SpacegroupAnalyzer(
            Structure.from_file(cif)
        ).get_conventional_standard_structure()

        _, z = s.composition.get_reduced_composition_and_factor()

        rows.append({
            "cif_file": cif,
            "Vol/Fu": s.volume / z,
        })

    except Exception as e:
        print(f"{cif}: {e}")

pd.DataFrame(rows).to_excel(
    "VolFu.xlsx",
    index=False
)

## Additional Features Required

This script generates 13 of the 18 features required by the FWHM model. To complete `To_predict_fwhm.xlsx`, the following descriptors must be added manually:

- `max_metal_ligand_bond_length` (from VESTA)
- `mean_metal_ligand_bond_length` (from VESTA)
- `polyhedron volume` (from VESTA)
- `x` (Cr³⁺ concentration)

The `R5` descriptor should be calculated using the ionic radii listed in the table provided in the README.